In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict, Literal, Annotated
from dotenv import load_dotenv
import time
import os

load_dotenv()

True

In [3]:
class CrashState(TypedDict):
    init_input: str
    step1: str
    step2: str
    step3: str

In [4]:
def step1(state: CrashState) -> str:
    print("Executing step 1")
    return {"step1": "done", "input": state["init_input"]}

def step2(state: CrashState) -> str:
    print("step 2 is hanging.... now manually kill the kernel to test crash prevention")
    time.sleep(3)
    return {"step2": "done"}

def step3(state: CrashState) -> str:
    print("Executing step 3")
    return {"step3": "done", "input": state["init_input"]}

In [5]:
graph = StateGraph(CrashState)
graph.add_node('step1', step1)
graph.add_node('step2', step2)
graph.add_node('step3', step3)

graph.add_edge(START, 'step1')
graph.add_edge('step1', 'step2')
graph.add_edge('step2', 'step3')
graph.add_edge('step3', END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [8]:
try:
    print("graph execution started... please interrupt the kernel within 20 seconds to test crash prevention")
    workflow.invoke(None, config={'configurable': {'thread_id': '1'}})
except KeyboardInterrupt:
    print(f"Workflow execution terminated due to keyboard interrupt")

graph execution started... please interrupt the kernel within 20 seconds to test crash prevention
step 2 is hanging.... now manually kill the kernel to test crash prevention
Executing step 3


In [9]:
list(
    workflow.get_state_history(
        {"configurable": {"thread_id": "1"}}
    )
)

[StateSnapshot(values={'init_input': 'test', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f168157-4bef-6e80-8003-eacda28e5acf'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-06-14T17:21:18.747161+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f168157-4be9-60b2-8002-7b211298656e'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'init_input': 'test', 'step1': 'done', 'step2': 'done'}, next=('step3',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f168157-4be9-60b2-8002-7b211298656e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-14T17:21:18.744280+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f168155-edb3-6a20-8001-fafb06c97826'}}, tasks=(PregelTask(id='9a11127d-a9c7-535f-879e-7a7454894653', 